# Calibration supervisee de Google TimesFM 2.5 (PyTorch)

**Seance 4 - Complement au TP 2** - Golden Collar

## Approche

Le full fine-tuning des **poids** de TimesFM exige son forward differentiable interne
(`model.model(...)`), specifique a la version de la librairie. Ici on adopte une approche
**robuste et reellement entrainable** :

1. **TimesFM reste gele** et fournit une prevision **zero-shot** (extraction de features).
2. On entraine une **petite tete de calibration differentiable** par-dessus, qui corrige
   la prevision de base. Le gradient circule correctement (seule la tete apprend).

> Note : c'est du fine-tuning **cote sortie** (output calibration). Pour entrainer les poids
> du backbone, il faudrait passer par `model.model(...)` (forward differentiable, dependant
> de la version timesfm) - hors perimetre ici.

> Avertissement donnees : un cours d'action est proche d'une **marche aleatoire** ; l'exemple
> sert a illustrer le pipeline, pas a produire un signal de trading.

---
## Partie 1 - Installation

In [ ]:
# A executer une seule fois (TimesFM 2.5 n'est pas sur PyPI) :
# !git clone https://github.com/google-research/timesfm.git
# !cd timesfm && pip install -e .[torch]
# !pip install yfinance pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)
torch.set_float32_matmul_precision("high")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch :", torch.__version__, "| device :", DEVICE)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0),
          f"| VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

---
## Partie 2 - Donnees et fenetres glissantes

In [ ]:
import yfinance as yf

TICKER, START, END = "AAPL", "2018-01-01", "2024-12-31"
df = yf.download(TICKER, start=START, end=END, auto_adjust=True)[['Close']].dropna()
df.columns = ['close']
values = df['close'].values.astype(np.float32)
print(f"{len(values)} jours | {df.index[0].date()} -> {df.index[-1].date()}")

plt.figure(figsize=(14, 4))
plt.plot(df.index, values, color='steelblue', lw=0.8)
plt.title(f'Cours de cloture - {TICKER}'); plt.ylabel('prix ($)'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
CONTEXT_LENGTH = 128     # contexte fourni a TimesFM
HORIZON_LENGTH = 32      # nombre de jours a prevoir
STRIDE         = 3       # pas entre fenetres (reduit le cout de precalcul)

def make_windows(data, ctx, hor, stride):
    n = (len(data) - ctx - hor) // stride + 1
    C = np.stack([data[i*stride : i*stride+ctx]               for i in range(n)])
    F = np.stack([data[i*stride+ctx : i*stride+ctx+hor]       for i in range(n)])
    return C.astype(np.float32), F.astype(np.float32)

C, F = make_windows(values, CONTEXT_LENGTH, HORIZON_LENGTH, STRIDE)

# Split CHRONOLOGIQUE des fenetres (train / val / test distincts -> pas de fuite)
n = len(C); i1, i2 = int(0.70*n), int(0.85*n)
C_tr, F_tr = C[:i1],   F[:i1]
C_va, F_va = C[i1:i2], F[i1:i2]
C_te, F_te = C[i2:],   F[i2:]
print(f"Fenetres : train {len(C_tr)} | val {len(C_va)} | test {len(C_te)}")

---
## Partie 3 - Chargement de TimesFM 2.5 (gele)

In [ ]:
import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(timesfm.ForecastConfig(
    max_context=CONTEXT_LENGTH, max_horizon=HORIZON_LENGTH,
    normalize_inputs=True, use_continuous_quantile_head=True,
    force_flip_invariance=True, infer_is_positive=True, fix_quantile_crossing=True,
))
print("TimesFM 2.5 charge | params :", f"{sum(p.numel() for p in model.parameters()):,}")

---
## Partie 4 - Previsions zero-shot (features geleees)

On precalcule la prevision zero-shot de TimesFM pour chaque fenetre. Ces sorties servent
d'entree (figee) a la tete de calibration. C'est l'etape la plus couteuse (une passe du
Transformer par lot).

In [ ]:
def tf_base_forecasts(model, contexts, horizon, batch=32):
    out = []
    for i in range(0, len(contexts), batch):
        chunk = contexts[i:i+batch]
        point, _ = model.forecast(horizon=horizon, inputs=[c.tolist() for c in chunk])
        out.append(np.asarray(point, dtype=np.float32)[:, :horizon])
    return np.concatenate(out, axis=0)

print("Precalcul des previsions zero-shot...")
base_tr = tf_base_forecasts(model, C_tr, HORIZON_LENGTH)
base_va = tf_base_forecasts(model, C_va, HORIZON_LENGTH)
base_te = tf_base_forecasts(model, C_te, HORIZON_LENGTH)
print("Termine.")

def metrics(actual, pred):
    actual, pred = np.asarray(actual), np.asarray(pred)
    return {'MAE':  float(np.mean(np.abs(actual - pred))),
            'RMSE': float(np.sqrt(np.mean((actual - pred) ** 2))),
            'MAPE': float(np.mean(np.abs((actual - pred) / actual)) * 100)}

m_zs = metrics(F_te, base_te)   # zero-shot sur le test
print("Zero-shot test :", {k: round(v, 3) for k, v in m_zs.items()})

---
## Partie 5 - Tete de calibration (differentiable, entrainable)

TimesFM est gele : `base` (sa prevision) est une **feature fixe**. La tete apprend une
**correction residuelle** ; le gradient ne traverse que la tete -> entrainement correct.

In [ ]:
class CalibrationHead(nn.Module):
    """corrige une prevision de base : sortie = base + MLP([base, dernier_point])."""
    def __init__(self, horizon, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(horizon + 1, hidden), nn.ReLU(),
            nn.Linear(hidden, horizon))

    def forward(self, base, last_value):
        return base + self.net(torch.cat([base, last_value], dim=1))


def to_tensors(base, C_ctx, F_fut):
    base = torch.tensor(base, dtype=torch.float32, device=DEVICE)
    last = torch.tensor(C_ctx[:, -1:], dtype=torch.float32, device=DEVICE)   # dernier point du contexte
    y    = torch.tensor(F_fut, dtype=torch.float32, device=DEVICE)
    return base, last, y

base_tr_t, last_tr_t, y_tr_t = to_tensors(base_tr, C_tr, F_tr)
base_va_t, last_va_t, y_va_t = to_tensors(base_va, C_va, F_va)

head = CalibrationHead(HORIZON_LENGTH).to(DEVICE)
opt  = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

best_val, best_state, patience, wait = float('inf'), None, 30, 0
hist = {'train': [], 'val': []}
for epoch in range(300):
    head.train(); opt.zero_grad()
    loss = loss_fn(head(base_tr_t, last_tr_t), y_tr_t)
    loss.backward(); opt.step()
    head.eval()
    with torch.no_grad():
        vl = loss_fn(head(base_va_t, last_va_t), y_va_t).item()
    hist['train'].append(loss.item()); hist['val'].append(vl)
    if vl < best_val:
        best_val, best_state, wait = vl, {k: v.clone() for k, v in head.state_dict().items()}, 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping epoch {epoch+1}"); break
head.load_state_dict(best_state)
print(f"Meilleure val MSE : {best_val:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(hist['train'], label='train'); plt.plot(hist['val'], label='val')
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('MSE (log)'); plt.legend(); plt.grid(alpha=.3)
plt.title('Tete de calibration'); plt.tight_layout(); plt.show()

---
## Partie 6 - Evaluation : zero-shot vs calibre

In [ ]:
head.eval()
with torch.no_grad():
    base_te_t, last_te_t, _ = to_tensors(base_te, C_te, F_te)
    cal_te = head(base_te_t, last_te_t).cpu().numpy()

m_cal = metrics(F_te, cal_te)

print(f"{'Metrique':<8}{'Zero-Shot':>12}{'Calibre':>12}{'Amelioration':>14}")
print('-' * 46)
for k in ['MAE', 'RMSE', 'MAPE']:
    imp = (m_zs[k] - m_cal[k]) / m_zs[k] * 100
    print(f"{k:<8}{m_zs[k]:>12.3f}{m_cal[k]:>12.3f}{imp:>+13.1f}%")

In [ ]:
# Visualisation sur une fenetre de test
j = len(C_te) // 2
plt.figure(figsize=(13, 5))
plt.plot(range(-CONTEXT_LENGTH, 0), C_te[j], color='gray', alpha=.5, label='contexte')
h = range(HORIZON_LENGTH)
plt.plot(h, F_te[j],   'k-o', ms=3, label='reel')
plt.plot(h, base_te[j], '--s', ms=3, color='#2196F3', label=f"zero-shot (MAE={metrics(F_te[j], base_te[j])['MAE']:.2f})")
plt.plot(h, cal_te[j],  '-^',  ms=3, color='#FF5722', label=f"calibre (MAE={metrics(F_te[j], cal_te[j])['MAE']:.2f})")
plt.axvline(0, color='gray', ls=':'); plt.ylabel('prix ($)'); plt.legend(); plt.grid(alpha=.3)
plt.title(f'TimesFM 2.5 : zero-shot vs calibre - {TICKER}'); plt.tight_layout(); plt.show()

---
## Partie 7 - A retenir

- **TimesFM gele = baseline zero-shot solide** ; la tete de calibration l'ajuste a la serie cible.
- Le gradient ne traverse **que** la tete : entrainement correct et leger (quelques milliers de params).
- Pour entrainer les **poids du backbone** (vrai full fine-tuning), il faut le forward
  differentiable `model.model(...)` (API specifique a la version timesfm) - ne pas passer par
  `model.forecast()` qui tourne en inference (no_grad, sortie numpy -> aucun gradient).

## References
- [TimesFM (GitHub)](https://github.com/google-research/timesfm)
- Das et al. (2024) - *A Decoder-only Foundation Model for Time-Series Forecasting* (ICML 2024)